# Figshare Document Pipeline

1. **DOI lookup** — batch-search Figshare for articles matching DOIs in `dois.txt`
2. **Article details** — fetch full metadata for each discovered article
3. **Local copy** — copy documents already present in the local backup
4. **Download** — download remaining documents directly from Figshare

In [ ]:
import json
import re
import tempfile
import time
from pathlib import Path

import fitz
import requests
import shutil
import subprocess
from tqdm.auto import tqdm

BASE_URL = "https://api.figshare.com/v2"
DOIS_FILE = Path("dois.txt")
assert DOIS_FILE.exists()

TRANSLATION_TABLE_FILE = Path("translation_table.json")
assert TRANSLATION_TABLE_FILE.exists()

METADATA_DIR = Path("figshare_metadata")
DETAILS_DIR = Path("figshare_details")
MISSING_DOIS_FILE = METADATA_DIR / "missing_dois.txt"
BATCH_SIZE = 10

PDF_DIR = Path("pdfs")
EXTRACTED_DIR = Path("extracted")
TEXT_DIR = EXTRACTED_DIR / "text"
BOLD_TEXT_DIR = EXTRACTED_DIR / "bold"
TEXT_DIR.mkdir(parents=True, exist_ok=True)
BOLD_TEXT_DIR.mkdir(parents=True, exist_ok=True)

DOCUMENT_EXTENSIONS = (".pdf", ".docx", ".doc")
MAX_RETRIES = 3
RETRY_DELAY = 5  # seconds
TIMEOUT = 60  # seconds

for dir in [METADATA_DIR, DETAILS_DIR, PDF_DIR, TEXT_DIR, BOLD_TEXT_DIR]:
    dir.mkdir(parents=True, exist_ok=True)

## 1. Load DOIs

In [ ]:
dois = [
    line.strip()
    for line in DOIS_FILE.read_text().splitlines()
    if line.strip() and not line.startswith("#")
]
print(f"Loaded {len(dois):,} DOIs from {DOIS_FILE}")

## 2. Batch search Figshare for metadata

In [ ]:
def _search_batch(doi_batch, page_size=100):
    """
    Search Figshare for articles whose related materials reference any DOI
    in *doi_batch*.  Returns the raw list of article search results.
    """
    query = " OR ".join(f":resource_doi: {d}" for d in doi_batch)
    data = {
        "search_for": query,
        "page_size": page_size,
        "order": "published_date",
        "order_direction": "desc",
    }
    resp = requests.post(f"{BASE_URL}/articles/search", json=data, timeout=TIMEOUT)
    resp.raise_for_status()
    return resp.json()


def get_metadata_path(doi):
    return METADATA_DIR / f"{doi}.json"


def retrieve_metadata(dois):
    n_batches = (len(dois) + BATCH_SIZE - 1) // BATCH_SIZE
    not_found = []
    for i in tqdm(range(0, len(dois), BATCH_SIZE), total=n_batches, desc="Metadata"):
        batch = dois[i : i + BATCH_SIZE]

        try:
            articles = _search_batch(batch)
        except requests.RequestException as exc:
            tqdm.write(f"  ERROR — {exc}")
            time.sleep(2)
            continue

        per_doi = {doi: [] for doi in batch}
        for art in articles:
            doi = art["resource_doi"]
            if doi in per_doi:
                per_doi[doi].append(art)

        for doi, arts in per_doi.items():
            if arts:
                file_path = get_metadata_path(doi)
                file_path.parent.mkdir(parents=True, exist_ok=True)
                with open(file_path, "w") as f:
                    json.dump(arts, f, indent=2)
            else:
                not_found.append(doi)

        time.sleep(1)

    return not_found

In [ ]:
missing_dois = (
    MISSING_DOIS_FILE.read_text().splitlines() if MISSING_DOIS_FILE.exists() else []
)
unseen_dois = [
    doi
    for doi in dois
    if doi not in missing_dois and not get_metadata_path(doi).exists()
]
not_found = retrieve_metadata(unseen_dois)
MISSING_DOIS_FILE.write_text("\n".join(sorted(missing_dois + not_found)))
print(
    f"Searched {len(unseen_dois):,} new DOIs, {len(unseen_dois) - len(not_found):,} found"
)
print(f"Metadata files: {len(list(METADATA_DIR.glob('**/*.json'))):,}")

## 3. Retrieve article details

In [ ]:
def collect_article_ids(type_filter="journal contribution"):
    """Return a sorted list of unique article IDs from all metadata files.

    If *type_filter* is set, only include articles whose
    ``defined_type_name`` matches.
    """
    ids = set()
    for path in METADATA_DIR.glob("**/*.json"):
        with open(path) as f:
            for art in json.load(f):
                if type_filter and art.get("defined_type_name") != type_filter:
                    continue
                ids.add(art["id"])
    return sorted(ids)


def fetch_article_detail(article_id):
    resp = requests.get(f"{BASE_URL}/articles/{article_id}", timeout=TIMEOUT)
    resp.raise_for_status()
    return resp.json()


def retrieve_details(article_ids):
    for aid in tqdm(article_ids, desc="Details"):
        dest = DETAILS_DIR / f"{aid}.json"
        if dest.exists():
            continue

        for attempt in range(MAX_RETRIES):
            try:
                detail = fetch_article_detail(aid)
                with open(dest, "w") as f:
                    json.dump(detail, f, indent=2)
                break
            except requests.RequestException as exc:
                tqdm.write(f"  {aid} attempt {attempt + 1} — {exc}")
                time.sleep(2 ** (attempt + 1))

        time.sleep(0.5)

In [ ]:
article_ids = collect_article_ids()
unseen_ids = [aid for aid in article_ids if not (DETAILS_DIR / f"{aid}.json").exists()]
print(f"{len(article_ids):,} unique articles, {len(unseen_ids):,} still need details")

retrieve_details(unseen_ids)
print(f"Detail files: {len(list(DETAILS_DIR.glob('*.json'))):,}")

## 4. Download documents from Figshare

In [ ]:
missing = []
already_downloaded = 0

for detail_file in sorted(DETAILS_DIR.glob("*.json")):
    with open(detail_file) as f:
        article = json.load(f)

    for file_entry in article.get("files", []):
        filename = file_entry["name"]
        if not filename.lower().endswith(DOCUMENT_EXTENSIONS):
            continue
        if (PDF_DIR / filename).exists():
            already_downloaded += 1
            continue
        missing.append(
            {
                "filename": filename,
                "download_url": file_entry["download_url"],
                "article_id": detail_file.stem,
            }
        )

print(f"Documents already in {PDF_DIR}/: {already_downloaded}")
print(f"Documents to download:           {len(missing):,}")

In [ ]:
downloaded = 0
failed = []

session = requests.Session()

for entry in tqdm(missing, desc="Downloading"):
    dest = PDF_DIR / entry["filename"]
    if dest.exists():
        downloaded += 1
        continue

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = session.get(entry["download_url"], timeout=TIMEOUT)
            resp.raise_for_status()
            with tempfile.NamedTemporaryFile(
                dir=PDF_DIR, delete=False, suffix=".tmp"
            ) as tmp:
                tmp.write(resp.content)
                tmp_path = Path(tmp.name)
            tmp_path.rename(dest)
            downloaded += 1
            break
        except Exception as exc:
            if attempt < MAX_RETRIES:
                time.sleep(RETRY_DELAY * attempt)
            else:
                failed.append({**entry, "error": str(exc)})

print(f"\nDownloaded: {downloaded}")
print(f"Failed:     {len(failed)}")

In [ ]:
if failed:
    print("Failed downloads:")
    for entry in failed:
        print(
            f"  {entry['filename']} (article {entry['article_id']}): {entry['error']}"
        )

## 5. Convert documents to PDF

In [ ]:
libreoffice_available = shutil.which("libreoffice") is not None

converted_from_word = 0

if libreoffice_available:
    doc_files = sorted(
        p for p in PDF_DIR.iterdir() if p.suffix.lower() in (".doc", ".docx")
    )
    print(f"Documents to convert: {len(doc_files)}")

    conv_failed = []

    for src in tqdm(doc_files, desc="Converting"):
        pdf_dest = src.with_suffix(".pdf")
        if pdf_dest.exists():
            converted_from_word += 1
            continue

        try:
            with tempfile.TemporaryDirectory(dir=PDF_DIR) as tmp_dir:
                result = subprocess.run(
                    [
                        "libreoffice",
                        "--headless",
                        "--convert-to",
                        "pdf",
                        "--outdir",
                        tmp_dir,
                        str(src),
                    ],
                    capture_output=True,
                    text=True,
                    timeout=120,
                )
                tmp_pdf = Path(tmp_dir) / (src.stem + ".pdf")
                if result.returncode == 0 and tmp_pdf.exists():
                    tmp_pdf.rename(pdf_dest)
                    src.unlink()
                    converted_from_word += 1
                else:
                    conv_failed.append((src.name, result.stderr.strip()))
        except Exception as exc:
            conv_failed.append((src.name, str(exc)))

    print(f"Converted: {converted_from_word}")
    print(f"Failed:    {len(conv_failed)}")
    if conv_failed:
        for name, err in conv_failed:
            print(f"  {name}: {err}")

else:
    print("libreoffice not found in PATH; install it to convert .doc/.docx files")

## 6. Convert documents to text and locate bold text blocks

In [ ]:
BOLD_FLAG = 1 << 4
COMMON_FRAG_PATTERN = re.compile(r"(yl|ox)")
STEREO_PATTERN = re.compile(
    r"""
    \(
        (?:\d+[RS]|[EZ])               # first descriptor
        (?:\s*,\s*(?:\d+[RS]|[EZ]))*   # optional additional ones
    \)
    """,
    re.VERBOSE,
)
TRANSLATION_TABLE = str.maketrans(json.load(open("translation_table.json")))


class BoldBlock(dict):
    def __init__(self, start, stop, full_text):
        super().__init__()
        self["type"] = "bold"
        self["start"] = start
        self["stop"] = stop
        self["text"] = full_text[start:stop]


def has_stereo(text):
    return bool(STEREO_PATTERN.search(text))


def has_common_fragment(text):
    return bool(COMMON_FRAG_PATTERN.search(text))


def is_worth_keeping(text):
    return has_stereo(text) or has_common_fragment(text)


def _is_bold(span):
    return span["flags"] & BOLD_FLAG or "bold" in span["font"].lower()


def _is_symbols(text):
    return all(not (c.isalnum() or c in " \n\f") for c in text)


def _add_text(text, full_text, pos):
    return full_text + text, pos + len(text)


def _add_char_if_needed(char, full_text, pos):
    if full_text and full_text[-1] != char:
        return full_text + char, pos + 1
    return full_text, pos


def get_text(pdf):
    full_text = ""
    pos = 0

    blocks = []
    bold_text = ""
    bold_start = bold_stop = None

    with fitz.open(str(pdf)) as doc:
        for page in doc:
            for block in page.get_text("dict")["blocks"]:
                for line in block.get("lines", []):
                    for span in line["spans"]:
                        text = span["text"].translate(TRANSLATION_TABLE)
                        if not text.isprintable():
                            continue
                        if _is_bold(span) or (bold_text and _is_symbols(text)):
                            if bold_text == "":
                                bold_start = pos
                            bold_text += text
                            bold_stop = pos + len(text)
                        elif bold_text != "":
                            if is_worth_keeping(bold_text):
                                blocks.append(
                                    BoldBlock(bold_start, bold_stop, full_text)
                                )
                            bold_text = ""
                        full_text, pos = _add_text(text, full_text, pos)
                    full_text, pos = _add_char_if_needed(" ", full_text, pos)
                full_text, pos = _add_char_if_needed(" ", full_text, pos)
            full_text, pos = _add_char_if_needed("\n", full_text, pos)

    if bold_text and is_worth_keeping(bold_text):
        blocks.append(BoldBlock(bold_start, bold_stop, full_text))

    return full_text, blocks

In [ ]:
def read_empty_files_list(file_path):
    if file_path.exists():
        with open(file_path) as f:
            return json.load(f)
    else:
        return []


def save_empty_files_list(file_path, missing):
    with open(file_path, "w") as f:
        json.dump(missing, f, indent=2)


empty_text_file_list = EXTRACTED_DIR / "empty_text.json"
empty_bold_file_list = EXTRACTED_DIR / "empty_bold.json"

empty_text_files = read_empty_files_list(empty_text_file_list)
empty_bold_files = read_empty_files_list(empty_bold_file_list)

num_empty_text_files = len(empty_text_files)
num_empty_bold_files = len(empty_bold_files)

pdf_files = sorted(PDF_DIR.glob("*.pdf"))
extracted = 0
for pdf in tqdm(pdf_files):
    text_file = TEXT_DIR / f"{pdf.stem}.txt"
    bold_file = BOLD_TEXT_DIR / f"{pdf.stem}.json"
    if (text_file.exists() or text_file.stem in empty_text_files) and (
        bold_file.exists() or bold_file.stem in empty_bold_files
    ):
        continue
    extracted += 1
    pdf_text, bold_blocks = get_text(pdf)
    if pdf_text:
        with open(text_file, "w") as f:
            f.write(pdf_text)
    else:
        empty_text_files.append(pdf.stem)
    if bold_blocks:
        with open(bold_file, "w") as f:
            json.dump(bold_blocks, f, indent=2)
    else:
        empty_bold_files.append(pdf.stem)
    if (len(empty_text_files) - num_empty_text_files) % 100 == 0:
        save_empty_files_list(empty_text_file_list, empty_text_files)
    if (len(empty_bold_files) - num_empty_bold_files) % 100 == 0:
        save_empty_files_list(empty_bold_file_list, empty_bold_files)

save_empty_files_list(empty_text_file_list, empty_text_files)
save_empty_files_list(empty_bold_file_list, empty_bold_files)

## 7. Summary

In [ ]:
print(f"DOIs loaded:    {len(dois):,}")
print(f"Metadata files: {len(list(METADATA_DIR.glob('**/*.json'))):,}")
print(f"Detail files:   {len(list(DETAILS_DIR.glob('*.json'))):,}")
print(f"PDF Documents:  {len(pdf_files)}")
print(f"Extracted text: {len(list(TEXT_DIR.glob('**/*.txt'))):,}")